# Intel® Extension for Scikit-learn Ridge Regression for Higgs dataset

In [1]:
from timeit import default_timer as timer
from sklearn import metrics
from sklearn.model_selection import train_test_split
import warnings
from sklearn.datasets import fetch_openml
from sklearn.preprocessing import LabelEncoder
from IPython.display import HTML

warnings.filterwarnings("ignore")

### Download the data

In [2]:
dataset = fetch_openml(data_id=45570, as_frame=True)

In [3]:
# Check the keys of the dataset to understand its structure
print(dataset.keys())

dict_keys(['data', 'target', 'frame', 'categories', 'feature_names', 'target_names', 'DESCR', 'details', 'url'])


In [3]:
import pandas as pd

# Access the data and target
x = dataset.data
y = dataset.target

# Check data types
print(x.dtypes)

# Select categorical columns based on data types
categorical_cols = x.select_dtypes(include=['object', 'category']).columns.tolist()
print(f'Categorical columns based on data types: {categorical_cols}')

# Select columns with a small number of unique values
low_cardinality_cols = [col for col in x.columns if x[col].nunique() < 10]
print(f'Columns with low cardinality: {low_cardinality_cols}')

# Combine both methods to get a comprehensive list of categorical columns
categorical_cols = list(set(categorical_cols + low_cardinality_cols))
print(f'Final list of categorical columns: {categorical_cols}')

lepton pT                   float64
lepton eta                  float64
lepton phi                  float64
missing energy magnitude    float64
missing energy phi          float64
jet 1 pt                    float64
jet 1 eta                   float64
jet 1 phi                   float64
jet 1 b-tag                 float64
jet 2 pt                    float64
jet 2 eta                   float64
jet 2 phi                   float64
jet 2 b-tag                 float64
jet 3 pt                    float64
jet 3 eta                   float64
jet 3 phi                   float64
jet 3 b-tag                 float64
jet 4 pt                    float64
jet 4 eta                   float64
jet 4 phi                   float64
jet 4 b-tag                 float64
m_jj                        float64
m_jjj                       float64
m_lv                        float64
m_jlv                       float64
m_bb                        float64
m_wbb                       float64
m_wwbb                      

### Preprocessing
Let's encode categorical features with LabelEncoder

In [5]:
# Access the data and target
for col in ['jet 3 b-tag', 'jet 1 b-tag', 'jet 4 b-tag', 'jet 2 b-tag']:
    le = LabelEncoder().fit(x[col])
    x[col] = le.transform(x[col])

In [6]:
x_train, x_test, y_train, y_test = train_test_split(x, y, test_size=0.1, random_state=0)
x_train.shape, x_test.shape, y_train.shape, y_test.shape

((9900000, 28), (1100000, 28), (9900000,), (1100000,))

In [7]:
from sklearn.preprocessing import MinMaxScaler, StandardScaler

scaler_x = MinMaxScaler()
scaler_y = StandardScaler()

In [8]:
y_train = y_train.to_numpy().reshape(-1, 1)
y_test = y_test.to_numpy().reshape(-1, 1)

scaler_x.fit(x_train)
x_train = scaler_x.transform(x_train)
x_test = scaler_x.transform(x_test)

scaler_y.fit(y_train)
y_train = scaler_y.transform(y_train).ravel()
y_test = scaler_y.transform(y_test).ravel()

In [9]:
from sklearnex import patch_sklearn

patch_sklearn()

Intel(R) Extension for Scikit-learn* enabled (https://github.com/uxlfoundation/scikit-learn-intelex)


In [10]:
from sklearn.linear_model import Ridge

params = {
    "alpha": 0.3,
    "fit_intercept": False,
    "random_state": 0,
    "copy_X": False,
}
start = timer()
model = Ridge(random_state=0).fit(x_train, y_train)
train_patched = timer() - start
f"Intel® extension for Scikit-learn time: {train_patched:.2f} s"

'Intel® extension for Scikit-learn time: 0.36 s'

In [11]:
y_predict = model.predict(x_test)
mse_metric_opt = metrics.mean_squared_error(y_test, y_predict)
f"Patched Scikit-learn MSE: {mse_metric_opt}"

'Patched Scikit-learn MSE: 0.9023910206230603'

In [12]:
from sklearnex import unpatch_sklearn

unpatch_sklearn()

In [13]:
from sklearn.linear_model import Ridge

start = timer()
model = Ridge(random_state=0).fit(x_train, y_train)
train_unpatched = timer() - start
f"Original Scikit-learn time: {train_unpatched:.2f} s"

'Original Scikit-learn time: 3.38 s'

In [14]:
y_predict = model.predict(x_test)
mse_metric_original = metrics.mean_squared_error(y_test, y_predict)
f"Original Scikit-learn MSE: {mse_metric_original}"

'Original Scikit-learn MSE: 0.9023910206230603'

In [15]:
HTML(
    f"<h3>Compare MSE metric of patched Scikit-learn and original</h3>"
    f"MSE metric of patched Scikit-learn: {mse_metric_opt} <br>"
    f"MSE metric of unpatched Scikit-learn: {mse_metric_original} <br>"
    f"Metrics ratio: {mse_metric_opt/mse_metric_original} <br>"
    f"<h3>With Scikit-learn-intelex patching you can:</h3>"
    f"<ul>"
    f"<li>Use your Scikit-learn code for training and prediction with minimal changes (a couple of lines of code);</li>"
    f"<li>Fast execution training and prediction of Scikit-learn models;</li>"
    f"<li>Get the similar quality</li>"
    f"<li>Get speedup in <strong>{(train_unpatched/train_patched):.1f}</strong> times.</li>"
    f"</ul>"
)

In [20]:
from sklearn.datasets import fetch_california_housing
from sklearn.model_selection import train_test_split
from sklearn.linear_model import ElasticNet
import time

# Fetch the dataset
data = fetch_california_housing()
x = data.data
y = data.target

# Split the data
x_train, x_test, y_train, y_test = train_test_split(x, y, test_size=0.1, random_state=0)

# Train an Elastic Net model
model = ElasticNet(alpha=1.0, l1_ratio=0.5, random_state=0)

start_time = time.time()
model.fit(x_train, y_train)
end_time = time.time()

print(f"Training completed in {end_time - start_time} seconds.")

Training completed in 0.016169309616088867 seconds.
